# 테이블 불러오기

In [0]:
kca_df = spark.table("silver.kca_info.kca_normalized2").dropDuplicates()
display(kca_df)

In [0]:
# # kca_normalized3 csv --> 테이블로 변환
# kca_df2 = spark.read.option("header", "true").csv("/Volumes/silver/kca_info/kca_normalized3/kca_normalized2_updated.csv")
# kca_df2.write.mode("overwrite").saveAsTable("silver.kca_info.kca_normalized3")
# display(kca_df2)

# ----------------------------------------------

In [0]:
display(
    kca_df.filter(kca_df["카테고리"] == "채소")
           .select("재료명", "세부속성")
           .distinct()
           .orderBy("재료명", "세부속성")
)

In [0]:
display(
    kca_df.select("카테고리").distinct().orderBy("카테고리")
)

In [0]:
from pyspark.sql.functions import when

kca_df2 = kca_df.withColumn(
    "카테고리",
    when(kca_df["카테고리"].isin("계란", "정육"), "축산물").otherwise(kca_df["카테고리"])
)
display(kca_df2)

In [0]:
from pyspark.sql.functions import when

kca_df2 = kca_df2.withColumn(
    "재료명",
    when(kca_df2["재료명"] == "달걀", "계란").otherwise(kca_df2["재료명"])
)
display(kca_df2)

In [0]:
from pyspark.sql.functions import when

kca_df2 = kca_df2.withColumn(
    "카테고리",
    when(kca_df2["카테고리"] == "버섯", "특용작물")
    .when(kca_df2["카테고리"] == "채소", "채소류")
    .otherwise(kca_df2["카테고리"])
)
display(kca_df2)

In [0]:
display(
    kca_df2.filter(kca_df2["카테고리"] == "축산물")
           .select("재료명", "세부속성")
           .distinct()
           .orderBy("재료명", "세부속성")
)

In [0]:
from pyspark.sql.functions import lit

kca_df2 = kca_df2.withColumn("등급", lit(None))

In [0]:
display(kca_df2)

In [0]:
from pyspark.sql.functions import when, regexp_extract, regexp_replace

# '축산물' 카테고리의 '세부속성'에서 괄호 안 단어를 '등급'에 입력
kca_df2 = kca_df2.withColumn(
    "등급",
    when(
        kca_df2["카테고리"] == "축산물",
        regexp_extract(kca_df2["세부속성"], r"\(([^)]+)\)", 1)
    ).otherwise(kca_df2["등급"])
)

# '축산물' 카테고리의 '세부속성'에서 괄호 및 괄호 안 텍스트 제거
kca_df2 = kca_df2.withColumn(
    "세부속성",
    when(
        kca_df2["카테고리"] == "축산물",
        regexp_replace(kca_df2["세부속성"], r"\s*\([^)]+\)", "")
    ).otherwise(kca_df2["세부속성"])
)

display(kca_df2)

In [0]:
from pyspark.sql.functions import when, col

kca_df2 = kca_df2.withColumn(
    "단위_문자",
    when(col("단위").contains("kg"), "kg").otherwise(col("단위_문자"))
)
display(
    kca_df2.filter(col("단위").contains("kg"))
)

In [0]:
display(
    kca_df2.select("단위_문자").distinct().orderBy("단위_문자")
)

In [0]:
from pyspark.sql.functions import lit

kca_df2 = kca_df2.withColumn("비고", lit(None))
display(kca_df2)

In [0]:
from pyspark.sql.functions import regexp_extract, regexp_replace, when

kca_df2 = kca_df2.withColumn(
    "비고",
    regexp_extract(col("세부속성"), r"\(([^)]+)\)", 1)
)

kca_df2 = kca_df2.withColumn(
    "세부속성",
    regexp_replace(col("세부속성"), r"\s*\([^)]+\)", "")
)

display(kca_df2)

In [0]:
from pyspark.sql.functions import when, col

target_attrs = [
    "정성식품 오징어젓",
    "임금님표 이천쌀",
    "임금님표 이천쌀 특등급",
    "철원 오대쌀",
    "한성 오징어젓갈",
    "스모크델리 훈제오리",
    "CJ 1등급 깨끗한 계란",
    "목초를 먹고 자란 건강한 닭이 낳은 달걀",
    "청정원 자유방목 동물복지 유정란",
    "1등급 다향 훈제오리",
    "청정원 행복놀이터 동물복지 유정란",
    "당찬진미 특등급",
    "당진 해나루쌀 특등급",
    "바른고을 의성진쌀",
    "불릴필요없는 현미",
    "청정원 동물복지 청정유정란",
    "풀무원 동물복지 목초란",
    "프리미엄 파타고니아 항공직송 생연어 필렛",
    "CJ 동물복지 유정란",
    "대왕님표 여주 진상미",
    "의성마늘 훈제오리 슬라이스",
    "항공직송 동원생연어",
    "96시간 숙성 현미",
    "풀무원 동물복지 유정란",
    "햇살드리 수향미",
    "허브를 담은 정다운 훈제오리"
]

kca_df2 = kca_df2.withColumn(
    "비고",
    when(col("세부속성").isin(target_attrs), col("세부속성")).otherwise(col("비고"))
).withColumn(
    "세부속성",
    when(col("세부속성").isin(target_attrs), col("재료명")).otherwise(col("세부속성"))
)


In [0]:
from pyspark.sql.functions import when, col

kca_df2 = kca_df2.withColumn(
    "비고",
    when(col("세부속성") == "노르웨이 생연어 필렛", "노르웨이").otherwise(col("비고"))
).withColumn(
    "세부속성",
    when(col("세부속성") == "노르웨이 생연어 필렛", "생연어 필렛").otherwise(col("세부속성"))
)

display(
    kca_df2.select("재료명", "세부속성","등급","비고").distinct()
)

In [0]:
from pyspark.sql.functions import when, col

kca_df2 = kca_df2.withColumn(
    "재료명",
    when(col("재료명") == "연어", "연어필렛").otherwise(col("재료명"))
)

display(kca_df2)

In [0]:
from pyspark.sql.functions import regexp_replace, when, col

kca_df2 = kca_df2.withColumn(
    "재료명",
    when(col("재료명") == "쇠고기", "소고기").otherwise(col("재료명"))
).withColumn(
    "세부속성",
    regexp_replace(col("세부속성"), r"^(돼지고기|쇠고기)\s*", "")
)

display(
    kca_df2.filter(col("카테고리") == "축산물")
    .select("재료명", "세부속성", "등급", "비고")
    .distinct()
    .orderBy("재료명", "세부속성", "등급", "비고")
)

In [0]:
display(kca_df2)

In [0]:
from pyspark.sql.functions import col, when, trim, lit, sum as spark_sum

# 모든 문자열 컬럼에 대해 빈 문자열 또는 공백을 null로 변환
string_cols = [f.name for f in kca_df2.schema.fields if f.dataType.simpleString() == 'string']

# null 처리 전 각 컬럼의 빈값/공백 개수 집계
null_counts = []
for c in string_cols:
    null_count = kca_df2.filter((col(c) == "") | (trim(col(c)) == "")).count()
    null_counts.append((c, null_count))

# null 처리
for c in string_cols:
    kca_df2 = kca_df2.withColumn(
        c,
        when((col(c) == "") | (trim(col(c)) == ""), lit(None)).otherwise(col(c))
    )

# null 처리된 값 개수 출력
for col_name, count in null_counts:
    if count > 0:
        print(f"컬럼 '{col_name}'에서 {count}개 값을 null로 처리함")

In [0]:
display(kca_df2.distinct().count()) # 642,087
# display(kca_df2.count()) # 642,087

In [0]:
kca_df2.write.mode("overwrite").saveAsTable("silver.kca_info.kca_normalized_fin")

## 테이블 csv로 저장(단일 파일로 저장-coalesce) ----------------

In [0]:
kca_df2.coalesce(1).write.mode("overwrite").option("header", "true").csv("/Volumes/silver/kca_info/kca_normalized_fin")